# Why does the hip limit the axis? Error geometry, bias and head shape

No training. Uses saved weights only. HFValid, 101 patients, each predicted by the fold model that held it out.

1. **Along / across the axis**: each hip and knee error vector is split into a part along the expert axis and a part
   across it. To first order, axis angle error ~ |across(hip error - knee error)| / femur length.
2. **Bias or scatter**: mean signed hip error (x lateral, y posterior, z proximal). The bias is removed using only the
   other folds, and we check whether the hip and axis errors shrink.
3. **Head shape**: sphere radius fitted on the reconstruction vs on the true mask.

**Inputs**: `preprocessed`, `run1` (split.json), `v3` output (v3_hfvalid_cv.csv, ckpt/attnC_*, swinC_*),
`lm3d` output (lm3d_fold*.pt), `lm2d` output (lm2d_fold*.pt).
**Accelerator: GPU T4.** About 15-20 min. Output: `/kaggle/working/axisdec/` (+ axisdec.zip).

In [ ]:
!pip install -q monai==1.6.1 einops

In [ ]:
import glob, json, re
from pathlib import Path
import numpy as np, pandas as pd, SimpleITK as sitk, torch
from scipy import ndimage
from scipy.stats import wilcoxon
from monai.networks.nets import AttentionUnet, SwinUNETR

def find(pat, required=True):
    hits = glob.glob(f"/kaggle/input/**/{pat}", recursive=True)
    assert hits or not required, f"{pat} not found under /kaggle/input: add the dataset that contains it"
    return Path(hits[0]).parent if hits else None
ROOT, SPLIT, RES, W = find("dataset.csv"), find("split.json"), find("v3_hfvalid_cv.csv"), find("attnC_fold0.pt")
VSDIN = find("prepare_log.csv", required=False)
FULL, VOX = (256, 128, 96), 2.0
device = "cuda" if torch.cuda.is_available() else "cpu"; amp = device == "cuda"
folds = json.load(open(SPLIT / "split.json")); ids = sorted(p for f in folds for p in f)
FOLD = {p: k for k, f in enumerate(folds) for p in f}
rd = lambda p: sitk.GetArrayFromImage(sitk.ReadImage(str(p)))
DRR = {p: [rd(ROOT / p / f"{v}.png").astype(np.float32)[::-1] / 65535.0 for v in ("ap", "lat")] for p in ids}
SEG = {p: rd(ROOT / p / "seg.nii.gz").astype(bool) for p in ids}
print(len(ids), "patients | VSD:", VSDIN, "| device", device)

In [ ]:
# extractor, offset, models: copied from train_v3
def surface_points(m):
    z, y, x = np.nonzero(m & ~ndimage.binary_erosion(m)); return np.stack([x, y, z], 1).astype(float)
def _sphere_lsq(p):
    s, *_ = np.linalg.lstsq(np.c_[2 * p, np.ones(len(p))], (p ** 2).sum(1), rcond=None)
    c = s[:3]; r2 = s[3] + c @ c; return (c, np.sqrt(r2)) if r2 > 0 else (None, None)
def fit_head(mask, prox=0.35, med=0.6, r_vox=(9, 14), n_iter=500, seed=0):
    p = surface_points(mask); z = p[:, 2]; p = p[z >= z.max() - prox * (z.max() - z.min())]
    xl, xh = p[:, 0].min(), p[:, 0].max(); p = p[p[:, 0] <= xl + med * (xh - xl)]
    rng = np.random.default_rng(seed); bn, bc, br = -1, None, None
    for _ in range(n_iter):
        c, r = _sphere_lsq(p[rng.choice(len(p), 4, replace=False)])
        if c is None or not (r_vox[0] <= r <= r_vox[1]): continue
        k = (np.abs(np.linalg.norm(p - c, axis=1) - r) < 2.0).sum()
        if k > bn: bn, bc, br = k, c, r
    for t in (2.0, 1.5, 1.0):
        inl = np.abs(np.linalg.norm(p - bc, axis=1) - br) < t; bc, br = _sphere_lsq(p[inl])
    return bc, br
def fit_knee(mask, dist_frac=0.30, win=2):
    z, y, x = np.nonzero(mask); sel = z <= z.min() + dist_frac * (z.max() - z.min())
    xs, ys, zs = x[sel], y[sel], z[sel]; zl = np.unique(zs)
    zc = zl[int(np.argmax([xs[zs == q].max() - xs[zs == q].min() for q in zl]))]
    m = np.abs(zs - zc) <= win
    return np.array([(xs[m].min() + xs[m].max()) / 2.0, ys[m].mean(), zc], float)
def drop_small(m, frac=0.05):
    lab, n = ndimage.label(m)
    if n <= 1: return m
    s = np.bincount(lab.ravel()); s[0] = 0; return np.isin(lab, np.nonzero(s >= frac * s.max())[0])
def expert(pid):
    lm = json.load(open(ROOT / pid / "landmarks.json"))["landmarks"]
    return np.array(lm["femurHead"]["voxel"]), np.array(lm["kneeCenter"]["voxel"])
KNEE_GT_RAW = {p: fit_knee(SEG[p]) for p in ids}
OFF = {f: np.mean([expert(p)[1] - KNEE_GT_RAW[p] for k, fl in enumerate(folds) if k != f for p in fl], 0) for f in range(5)}
OFF_ALL = np.mean([expert(p)[1] - KNEE_GT_RAW[p] for p in ids], 0)

def _remap(k):
    k = k.replace("module.", ""); k = re.sub(r"\bsub0\b", "submodule.0", k); k = re.sub(r"\bsub1\b", "submodule.1", k)
    return re.sub(r"\bsubconv\b", "submodule.conv", k)
make = {"attnC": lambda: AttentionUnet(spatial_dims=3, in_channels=2, out_channels=1, channels=(16, 32, 64, 128), strides=(2, 2, 2)),
        "swinC": lambda: SwinUNETR(in_channels=2, out_channels=1, feature_size=48, use_checkpoint=True, spatial_dims=3)}
NETS = {}
for k in make:
    for f in range(5):
        n = make[k](); n.load_state_dict({_remap(a): b for a, b in torch.load(W / f"{k}_fold{f}.pt", map_location=device).items()})
        NETS[(k, f)] = n.to(device).eval()
norm = lambda im: ((im - im.mean()) / (im.std() + 1e-6)).astype(np.float32)
def predict(k, ap, lat, fold=None):
    ap, lat = norm(ap), norm(lat)
    x = torch.from_numpy(np.stack([np.broadcast_to(ap[:, None, :], FULL), np.broadcast_to(lat[:, :, None], FULL)]).astype(np.float32))[None].to(device)
    with torch.no_grad(), torch.autocast(device_type="cuda", enabled=amp):
        nets = [NETS[(k, fold)]] if fold is not None else [NETS[(k, f)] for f in range(5)]
        p = torch.stack([torch.sigmoid(n(x)) for n in nets]).mean(0)
    return drop_small((p > 0.5)[0, 0].float().cpu().numpy() > 0)
print("models loaded:", len(NETS))

In [ ]:
# 1) predicted masks (held-out fold model) and all landmark estimates
import time, shutil
import torch.nn.functional as F
from monai.networks.nets import UNet
OUT = Path("/kaggle/working/axisdec"); OUT.mkdir(parents=True, exist_ok=True)
W3, W2 = find("lm3d_fold0.pt"), find("lm2d_fold0.pt")
t0 = time.time()
PRED = {k: {p: predict(k, *DRR[p], fold=FOLD[p]) for p in ids} for k in make}
del NETS; torch.cuda.empty_cache(); print(f"masks predicted in {(time.time()-t0)/60:.1f} min")

def decode(prob, r=3):                             # argmax + probability-weighted centroid, any dimension
    out = []
    for p in prob:
        c = np.unravel_index(p.argmax(), p.shape)
        sl = tuple(slice(max(ci - r, 0), min(ci + r + 1, n)) for ci, n in zip(c, p.shape))
        w = p[sl]; g = np.mgrid[sl]; out.append(np.array([(w * gi).sum() / w.sum() for gi in g]))
    return out                                     # array order: 3D (z, y, x), 2D (row, col)
def load_lm3d(f):
    n = UNet(spatial_dims=3, in_channels=1, out_channels=2, channels=(8, 16, 32, 64, 128), strides=(2, 2, 2, 2), num_res_units=1)
    n.load_state_dict(torch.load(W3 / f"lm3d_fold{f}.pt", map_location=device)); return n.to(device).eval()
def load_lm2d(f):
    s = torch.load(W2 / f"lm2d_fold{f}.pt", map_location=device); out = []
    for v in ("ap", "lat"):
        n = UNet(spatial_dims=2, in_channels=1, out_channels=2, channels=(16, 32, 64, 128, 256), strides=(2, 2, 2, 2), num_res_units=2)
        n.load_state_dict(s[v]); out.append(n.to(device).eval())
    return out
LM3 = {f: load_lm3d(f) for f in range(5)}; LM2 = {f: load_lm2d(f) for f in range(5)}
def learned3d(net, mask):
    x = torch.from_numpy(mask[None, None].astype(np.float32)).to(device)
    with torch.no_grad(): p = F.softmax(net(x).float().flatten(2), -1).reshape(2, *FULL).cpu().numpy()
    return [d[::-1] for d in decode(p)]            # -> (x, y, z)
def heat2(net, im):
    x = torch.from_numpy(norm(im))[None, None].to(device)
    with torch.no_grad(): return F.softmax(net(x).flatten(2), -1).reshape(2, *im.shape).cpu().numpy()
def learned2d(nets, ap, lat):                      # AP pixel = (z, x), LAT pixel = (z, y)
    (hza, hx), (kza, kx) = decode(heat2(nets[0], ap)); (hzl, hy), (kzl, ky) = decode(heat2(nets[1], lat))
    return np.array([hx, hy, (hza + hzl) / 2]), np.array([kx, ky, (kza + kzl) / 2])

rows = []
for p in ids:
    f = FOLD[p]; eh, ek = expert(p)
    est = {}
    for src_, m in [("true", SEG[p])] + [(k, PRED[k][p]) for k in make]:
        (h, r) = fit_head(m); est[(src_, "rule")] = (h, fit_knee(m) + OFF[f], r)
        h3, k3 = learned3d(LM3[f], m); est[(src_, "learned")] = (h3, k3, np.nan)
    h2, k2 = learned2d(LM2[f], *DRR[p]); est[("images", "2D")] = (h2, k2, np.nan)
    for (src_, ex), (h, k, r) in est.items():
        rows.append(dict(case=p, fold=f, source=src_, extractor=ex, radius_mm=r * VOX,
                         **{f"{n}_{a}": v for n, vec in (("h", h), ("k", k), ("eh", eh), ("ek", ek)) for a, v in zip("xyz", vec * VOX)}))
X = pd.DataFrame(rows); X.to_csv(OUT / "landmarks_mm.csv", index=False)
print(len(X), "rows saved", f"| {(time.time()-t0)/60:.1f} min")

In [ ]:
# 2) along / across the expert axis
ang3 = lambda a, b: float(np.degrees(np.arccos(np.clip(a @ b / np.linalg.norm(a) / np.linalg.norm(b), -1, 1))))
V = lambda d, n: d[[f"{n}_x", f"{n}_y", f"{n}_z"]].to_numpy(float)
def geometry(d):
    h, k, eh, ek = V(d, "h"), V(d, "k"), V(d, "eh"), V(d, "ek")
    ax = eh - ek; L = np.linalg.norm(ax, axis=1); u = ax / L[:, None]
    split = lambda e: (np.abs((e * u).sum(1)), np.linalg.norm(e - (e * u).sum(1)[:, None] * u, axis=1))
    eH, eK = h - eh, k - ek; (ha, hp), (ka, kp) = split(eH), split(eK); _, dp = split(eH - eK)
    ang = np.array([ang3(a, b) for a, b in zip(h - k, ax)])
    return dict(hip=np.linalg.norm(eH, axis=1), hip_along=ha, hip_across=hp, knee=np.linalg.norm(eK, axis=1),
                knee_along=ka, knee_across=kp, across_diff=dp, approx_deg=np.degrees(dp / L), angle_deg=ang, L=L)
G = {key: geometry(d.sort_values("case")) for key, d in X.groupby(["source", "extractor"])}
order = [("true", "rule"), ("true", "learned"), ("attnC", "rule"), ("attnC", "learned"), ("swinC", "rule"), ("swinC", "learned"), ("images", "2D")]
md_ = lambda v: f"{np.median(v):.2f}"
print("median (mm unless noted)      hip  along across | knee along across | across(hip-knee)  approx deg  axis deg")
for key in order:
    g = G[key]
    print(f"{key[0]:6} {key[1]:8}            {md_(g['hip'])}  {md_(g['hip_along'])}  {md_(g['hip_across'])}  | {md_(g['knee'])}  "
          f"{md_(g['knee_along'])}  {md_(g['knee_across'])}  |  {md_(g['across_diff'])}          {md_(g['approx_deg'])}       {md_(g['angle_deg'])}")
g = G[("attnC", "rule")]
print(f"\ncheck: approx vs true angle, Spearman {pd.Series(g['approx_deg']).corr(pd.Series(g['angle_deg']), method='spearman'):.3f};"
      f" femur length median {np.median(g['L']):.0f} mm")
print("\nshare of squared error across the axis (sum across^2 / sum total^2):")
for key in order:
    g = G[key]
    print(f"  {key[0]:6} {key[1]:8} hip {np.sum(g['hip_across']**2)/np.sum(g['hip']**2):.2f}  knee {np.sum(g['knee_across']**2)/np.sum(g['knee']**2):.2f}")
print("\npaired, learned - rule (Wilcoxon, unadjusted):")
for s in ("true", "attnC", "swinC"):
    for m in ("hip_across", "knee_across", "knee_along"):
        d = G[(s, "learned")][m] - G[(s, "rule")][m]
        print(f"  {s:6} {m:12} median diff {np.median(d):+.2f} mm  p {wilcoxon(d).pvalue:.3g}")

In [ ]:
# 3) bias or scatter: mean signed hip error, then remove it using only the other folds
rng = np.random.default_rng(0)
print("mean signed hip error (mm) [95% bootstrap CI]; x lateral, y posterior, z proximal (canonical left femur)")
cal = []
for key in order:
    d = X[(X.source == key[0]) & (X.extractor == key[1])].sort_values("case")
    e = V(d, "h") - V(d, "eh"); f = d.fold.to_numpy()
    bs = np.array([e[rng.integers(0, len(e), len(e))].mean(0) for _ in range(5000)])
    lo, hi = np.percentile(bs, [2.5, 97.5], 0); mu = e.mean(0)
    ci = "  ".join(f"{a} {m:+.2f} [{l:+.2f}, {h_:+.2f}]" for a, m, l, h_ in zip("xyz", mu, lo, hi))
    rms = np.sqrt((e ** 2).sum(1).mean())
    print(f"  {key[0]:6} {key[1]:8} {ci}   |mean| {np.linalg.norm(mu):.2f} vs RMS {rms:.2f}")
    # cross-validated bias removal: for each fold, subtract the mean error of the other folds
    b = np.stack([e[f != k].mean(0) for k in f]); hc = V(d, "h") - b
    ax = V(d, "eh") - V(d, "ek"); kk = V(d, "k")
    ang_c = [ang3(a, c) for a, c in zip(hc - kk, ax)]
    cal.append(dict(source=key[0], extractor=key[1], hip_before=np.median(np.linalg.norm(e, axis=1)),
                    hip_after=np.median(np.linalg.norm(hc - V(d, "eh"), axis=1)),
                    axis_before=np.median(G[key]["angle_deg"]), axis_after=np.median(ang_c),
                    p_hip=wilcoxon(np.linalg.norm(hc - V(d, "eh"), axis=1) - np.linalg.norm(e, axis=1)).pvalue,
                    p_axis=wilcoxon(np.array(ang_c) - G[key]["angle_deg"]).pvalue))
C = pd.DataFrame(cal); C.to_csv(OUT / "bias_removal.csv", index=False)
print("\nafter removing the hip bias (estimated on the other folds), medians:")
print(C.round(3).to_string(index=False))

In [ ]:
# 4) head shape: fitted sphere radius on reconstructions vs true masks (rule extractor)
R = {s: X[(X.source == s) & (X.extractor == "rule")].sort_values("case").radius_mm.to_numpy() for s in ("true", "attnC", "swinC")}
print(f"true-mask head radius: median {np.median(R['true']):.2f} mm [{np.percentile(R['true'], 25):.2f}, {np.percentile(R['true'], 75):.2f}]")
for s in ("attnC", "swinC"):
    d = R[s] - R["true"]
    print(f"{s}: radius {np.median(R[s]):.2f} mm, paired diff median {np.median(d):+.2f} mm "
          f"[{np.percentile(d, 25):+.2f}, {np.percentile(d, 75):+.2f}], mean {d.mean():+.2f}, p {wilcoxon(d).pvalue:.3g};"
          f" Spearman |radius diff| vs hip error {pd.Series(np.abs(d)).corr(pd.Series(G[(s, 'rule')]['hip']), method='spearman'):.2f}")

import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 3, figsize=(12, 3.6))
for a, (s, c) in zip(ax[:2], (("attnC", "C0"), ("swinC", "C1"))):
    d = X[(X.source == s) & (X.extractor == "rule")]; e = V(d, "h") - V(d, "eh")
    a.scatter(e[:, 0], e[:, 2], s=8, c=c); a.axhline(0, c="k", lw=0.5); a.axvline(0, c="k", lw=0.5)
    a.plot(*e.mean(0)[[0, 2]], "k+", ms=14, mew=2); a.set_xlabel("x lateral (mm)"); a.set_ylabel("z proximal (mm)"); a.set_title(f"{s} hip error, coronal")
    a.set_aspect("equal")
for s in ("attnC", "swinC"): ax[2].scatter(R["true"], R[s], s=8, label=s)
lim = [R["true"].min() - 1, R["true"].max() + 1]; ax[2].plot(lim, lim, "k--", lw=0.8)
ax[2].set_xlabel("radius, true mask (mm)"); ax[2].set_ylabel("radius, reconstruction (mm)"); ax[2].legend()
plt.tight_layout(); plt.savefig(OUT / "hip_geometry.png", dpi=150); plt.show()
shutil.make_archive("/kaggle/working/axisdec", "zip", OUT); print("saved /kaggle/working/axisdec.zip")